[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prompt-engineering-certified/notebooks/day-06-advanced-techniques.ipynb#scrollTo=aa1bb2cc)

---
# Day 6 · Advanced Techniques — ReAct, Tree of Thought, and Meta-Prompting
**certified-journeys / prompt-engineering-certified** · Day 6 · Practice

> **Goal for today:** Implement the ReAct reasoning loop, a simplified Tree of Thought search, and a meta-prompt that generates prompts — the three techniques that underpin most production agent systems.


In [ ]:
%pip install -q openai


## Step 1 · Mock client and tool infrastructure

Advanced techniques like ReAct require a mock **tool** (e.g., search) that the model can call. We'll simulate a search engine that returns deterministic results.

| Technique | Core idea | Where it's used |
|---|---|---|
| ReAct | Thought → Action → Observation loop | Agents, Q&A with tools |
| Tree of Thought (ToT) | Branch + score + select at each step | Complex planning, multi-step reasoning |
| Meta-prompting | Prompt that generates prompts | Rapid task adaptation, prompt libraries |


In [ ]:
import json
import re
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

# ── Minimal mock LLM ───────────────────────────────────────────
@dataclass
class MockMessage:
    content: str
    role: str = "assistant"

@dataclass
class MockChoice:
    message: MockMessage
    index: int = 0

@dataclass
class MockCompletion:
    choices: List[MockChoice]
    model: str = "gpt-4o-mini"

MODEL = "gpt-4o-mini"

# Registry: last-added key wins; call register() before each test
_LLM_REGISTRY = {}

def register(key: str, response: str):
    _LLM_REGISTRY[key.lower()] = response

class MockOpenAI:
    class _Chat:
        class _Completions:
            def create(self, model, messages, **kwargs):
                last = messages[-1]["content"].lower()
                for k, v in _LLM_REGISTRY.items():
                    if k in last:
                        return MockCompletion([MockChoice(MockMessage(v))])
                return MockCompletion([MockChoice(MockMessage("[no mock matched]"))])
        completions = _Completions()
    chat = _Chat()

client = MockOpenAI()
print("Mock LLM client ready")


### What just happened?
- The registry is a simple `dict` — last-registered key wins, so tests can override each other cleanly.
- `register(key, response)` is called before each technique section to inject the right canned outputs.
- In production, replace `MockOpenAI` with `openai.OpenAI(api_key=...)` and use `gpt-4o-mini`.


## Step 2 · Mock search tool

ReAct requires at least one tool the model can call. We mock a **web search** that returns deterministic snippets. In production this would call a real search API (Bing, SerpAPI, Tavily, etc.).


In [ ]:
# ── Mock search tool ────────────────────────────────────────────

_SEARCH_DB = {
    "population of france": "France has a population of approximately 68 million people as of 2023.",
    "capital of france": "The capital of France is Paris.",
    "eiffel tower height": "The Eiffel Tower is 330 meters tall including its antenna.",
    "largest city france": "The largest city in France is Paris, with approximately 2.1 million residents in the city proper.",
    "france gdp": "France's GDP was approximately $2.78 trillion USD in 2022, making it the 7th largest economy in the world.",
    "python language creator": "Python was created by Guido van Rossum and first released in 1991.",
    "openai founded": "OpenAI was founded in December 2015 by Sam Altman, Elon Musk, and others.",
    "react prompting": "ReAct is a prompting technique that interleaves reasoning (Thought) and action steps to solve tasks.",
}

def search(query: str) -> str:
    """Mock search tool. Production equivalent: requests to Tavily or SerpAPI."""
    query_lower = query.lower().strip()
    # Substring match
    for key, result in _SEARCH_DB.items():
        if any(word in query_lower for word in key.split() if len(word) > 3):
            return result
    return f"No results found for '{query}'. Try a different query."

# Test the search tool
print("Tool: search('population of France')")
print("→", search("population of France"))
print()
print("Tool: search('Eiffel Tower height')")
print("→", search("Eiffel Tower height"))


### What just happened?
- **Substring matching** makes the mock tolerant of paraphrase — the model doesn't need exact search terms.
- The production equivalent is a `requests.get` to Tavily, SerpAPI, or Bing Search API.
- The tool returns a **string observation** — this is what gets fed back to the model in the next ReAct step.


## Step 3 · ReAct: Thought → Action → Observation loop

ReAct (Reasoning + Acting) interleaves:
1. **Thought**: the model reasons about what to do next
2. **Action**: the model calls a tool
3. **Observation**: the tool result is fed back
4. Repeat until the model produces a **Final Answer**

The `Thought:` step forces the model to plan **before** acting — without it, tool calls are random.


In [ ]:
# ── ReAct implementation ────────────────────────────────────────

REACT_SYSTEM = """\
You are a helpful assistant. To answer questions, use the following format:

Thought: [your reasoning about what to do next]
Action: search[<query>]
Observation: [tool result — provided by the system]

Repeat Thought/Action/Observation as needed.
When you have enough information, write:
Final Answer: [your complete answer]

Only call 'search' as your action. Never fabricate observations.
"""

# ── Canned ReAct model responses for each step ─────────────────
# Step 1: model decides to search
register("what is the population",
    "Thought: I need to find the population of France. I'll search for it.\n"
    "Action: search[population of France]")

# Step 2: after observation, model either searches again or answers
register("observation: france has a population",
    "Thought: I now know France's population is approximately 68 million.\n"
    "Final Answer: The population of France is approximately 68 million people (as of 2023).")


def parse_react_step(text: str) -> Tuple[str, Optional[str], Optional[str]]:
    """Parse a ReAct model output into (thought, action_query, final_answer)."""
    thought = ""
    action_query = None
    final_answer = None

    # Extract Thought
    thought_match = re.search(r"Thought:\s*(.+?)(?=\nAction:|\nFinal Answer:|$)", text, re.DOTALL)
    if thought_match:
        thought = thought_match.group(1).strip()

    # Extract Action
    action_match = re.search(r"Action:\s*search\[(.+?)\]", text)
    if action_match:
        action_query = action_match.group(1).strip()

    # Extract Final Answer
    answer_match = re.search(r"Final Answer:\s*(.+)$", text, re.DOTALL)
    if answer_match:
        final_answer = answer_match.group(1).strip()

    return thought, action_query, final_answer


def react_agent(question: str, max_steps: int = 5) -> str:
    """Run a ReAct loop until Final Answer or max_steps."""
    messages = [
        {"role": "system", "content": REACT_SYSTEM},
        {"role": "user", "content": question},
    ]

    print(f"Question: {question}")
    print("=" * 55)

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages)
        reply = response.choices[0].message.content

        thought, action_query, final_answer = parse_react_step(reply)

        print(f"[Step {step}]")
        if thought:
            print(f"  Thought    : {thought}")

        if final_answer:
            print(f"  Final Answer: {final_answer}")
            return final_answer

        if action_query:
            observation = search(action_query)
            print(f"  Action     : search[{action_query}]")
            print(f"  Observation: {observation}")

            # Append to conversation so the model sees the observation
            messages.append({"role": "assistant", "content": reply})
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        else:
            print(f"  [No action extracted — ending loop]")
            break

    return "[Max steps reached without final answer]"


answer = react_agent("What is the population of France?")


### What just happened?
- **Thought** makes the model commit to a plan before calling the tool — prevents random tool calls.
- The **observation** is appended to the conversation as a `user` message — the model then re-reasons.
- `parse_react_step` uses regex to extract the structured parts from free-text model output.
- **In production**: use function calling for tool invocation instead of parsing text — more reliable.


## Step 4 · Tree of Thought (ToT): branch, score, select

Tree of Thought generates **multiple candidate next steps**, scores each one, picks the best, and repeats. Unlike ReAct's linear chain, ToT explores a search tree — essential for tasks where the first idea isn't always the best.

Simplified ToT loop:
1. Generate N candidate next steps
2. Score each candidate (0–10) with a separate LLM call
3. Select the highest-scoring candidate
4. Repeat from step 1 with the selected step as context


In [ ]:
# ── Tree of Thought: simplified implementation ──────────────────

# Register mock responses for candidate generation
register("generate 3 candidate next steps",
    "1. Research the target audience demographics and pain points\n"
    "2. Write a one-sentence value proposition for the product\n"
    "3. Identify 3 competing products and their weaknesses")

# Register scoring responses — one per candidate
_SCORING_RESPONSES = {
    "research the target audience": "Score: 8\nReasoning: Audience research is foundational — without it, all subsequent decisions are guesses.",
    "one-sentence value proposition": "Score: 6\nReasoning: Useful but premature without knowing the audience first.",
    "3 competing products": "Score: 7\nReasoning: Competitive analysis helps differentiate but should follow audience research.",
}


def generate_candidates(task: str, context: str, n: int = 3) -> List[str]:
    """Generate N candidate next steps for a task."""
    prompt = (
        f"Task: {task}\n"
        f"Context so far: {context or 'None'}\n\n"
        f"Generate {n} candidate next steps. Number them 1, 2, 3."
    )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = resp.choices[0].message.content
    # Parse numbered list
    candidates = re.findall(r"\d+\.\s*(.+)", raw)
    return candidates[:n]


def score_candidate(candidate: str, task: str) -> Tuple[int, str]:
    """Score a single candidate step 0–10 and return (score, reasoning)."""
    # Use canned responses keyed by candidate text
    for key, response in _SCORING_RESPONSES.items():
        if key.lower() in candidate.lower():
            score_match = re.search(r"Score:\s*(\d+)", response)
            reason_match = re.search(r"Reasoning:\s*(.+)", response)
            score = int(score_match.group(1)) if score_match else 5
            reason = reason_match.group(1) if reason_match else ""
            return score, reason
    return 5, "No specific reasoning available"


def tot_step(task: str, context: str = "", n_candidates: int = 3) -> Tuple[str, int, List[dict]]:
    """Run one ToT step: generate → score → select best candidate."""
    candidates = generate_candidates(task, context, n=n_candidates)
    scored = []
    for c in candidates:
        score, reason = score_candidate(c, task)
        scored.append({"candidate": c, "score": score, "reason": reason})

    # Select best
    best = max(scored, key=lambda x: x["score"])
    return best["candidate"], best["score"], scored


# ── Run 2 ToT steps on a product strategy task ─────────────────
TASK = "Create a go-to-market strategy for a new AI-powered writing assistant"
context = ""

print(f"Task: {TASK}")
print("=" * 60)

for step_num in range(1, 3):
    best_candidate, best_score, all_scored = tot_step(TASK, context, n_candidates=3)

    print(f"\n[ToT Step {step_num}] Candidates:")
    for item in all_scored:
        marker = "→ SELECTED" if item["candidate"] == best_candidate else "  "
        print(f"  {marker} [{item['score']}/10] {item['candidate']}")
        print(f"           Reason: {item['reason']}")

    print(f"\n  Best step: {best_candidate} (score: {best_score}/10)")
    context = f"Step {step_num}: {best_candidate}"  # carry forward into next iteration


### What just happened?
- **Branching**: we generate 3 candidates instead of committing to the first idea.
- **Scoring**: a separate LLM call evaluates each candidate — explicit criteria beat implicit assumptions.
- **Selection**: we take the max-score candidate and carry it into the next step as context.
- ToT is 3–5× more expensive than a linear chain but finds better solutions on complex planning tasks.


## Step 5 · Meta-prompting: a prompt that writes prompts

A **meta-prompt** is a prompt that takes a task description as input and produces a high-quality prompt as output. This is useful for:
- Rapidly adapting a prompt template to new domains
- Building prompt libraries where humans describe intent and the LLM writes the technical prompt
- Bootstrapping few-shot examples automatically


In [ ]:
# ── Meta-prompt implementation ──────────────────────────────────

META_PROMPT_TEMPLATE = """\
You are an expert prompt engineer. Your job is to write a high-quality prompt for the task described below.

Rules for the prompt you write:
1. Include a clear role/persona at the start
2. Specify the exact output format
3. Include at least one constraint (length, tone, or style)
4. Add one few-shot example if the task benefits from it
5. End with a concrete instruction for the model

Task description: {task_description}

Write the prompt below. Start directly with the prompt text — do not explain or preamble.
"""

# Canned meta-prompt outputs
register("customer complaint email",
    """You are a professional customer service specialist with 10 years of experience resolving product issues.

Your task: Write a response to the customer complaint email provided below.

Output format:
- Greeting (1 sentence)
- Acknowledgement of the issue (1–2 sentences)
- Resolution or next steps (2–3 sentences)
- Closing with contact details (1 sentence)

Constraints:
- Maximum 150 words
- Empathetic tone — never defensive
- Never promise refunds unless the policy explicitly allows it

Example:
Customer: "My order arrived broken. This is unacceptable."
Response: "Dear [Name], Thank you for reaching out. We sincerely apologize that your order arrived damaged — this is not the experience we want for you. We will ship a replacement within 2 business days at no additional charge. If you have further questions, please contact us at support@example.com."

Now write a response to this customer complaint email:
{complaint_email}""")


def generate_prompt(task_description: str) -> str:
    """Use the meta-prompt to generate a specialized prompt for a task."""
    meta_input = META_PROMPT_TEMPLATE.format(task_description=task_description)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": meta_input}]
    )
    return response.choices[0].message.content


task_desc = "Respond to a customer complaint email about a damaged product"
generated_prompt = generate_prompt(task_desc)

print("Task description:", task_desc)
print("=" * 60)
print("Generated prompt:")
print("-" * 60)
print(generated_prompt)


### What just happened?
- The **meta-prompt** enforces structural rules: role, format, constraint, example, instruction — every time.
- The output is a **ready-to-use** prompt with a `{complaint_email}` placeholder — not a vague description.
- Meta-prompting is how you scale prompt authorship across a team without every engineer learning prompt engineering from scratch.


## Step 6 · Evaluating the meta-prompt output

A generated prompt needs evaluation — just like any other prompt. We use a simple rubric scorer to assess the output prompt quality.


In [ ]:
# ── Meta-prompt output evaluator ────────────────────────────────

QUALITY_RUBRIC = [
    ("has_role",       lambda p: any(w in p.lower() for w in ["you are", "you're", "as a", "act as"]),
     "Prompt includes a role/persona"),
    ("has_format",     lambda p: any(w in p.lower() for w in ["format", "output", "structure", "write"]),
     "Prompt specifies output format"),
    ("has_constraint", lambda p: any(w in p.lower() for w in ["maximum", "minimum", "no more", "only", "never", "always", "constraint"]),
     "Prompt includes at least one constraint"),
    ("has_example",    lambda p: any(w in p.lower() for w in ["example", "e.g.", "for instance", "sample"]),
     "Prompt includes a few-shot example"),
    ("has_instruction", lambda p: "{" in p and "}" in p,
     "Prompt has a placeholder for the actual input"),
]


def evaluate_prompt_quality(prompt: str) -> dict:
    """Score a generated prompt against the quality rubric."""
    results = {}
    for name, check_fn, description in QUALITY_RUBRIC:
        passed = check_fn(prompt)
        results[name] = {"passed": passed, "description": description}
    total = sum(1 for r in results.values() if r["passed"])
    return {"score": total, "max": len(QUALITY_RUBRIC), "checks": results}


quality = evaluate_prompt_quality(generated_prompt)
print(f"Meta-prompt output quality: {quality['score']}/{quality['max']}")
print("-" * 50)
for name, result in quality["checks"].items():
    status = "✓" if result["passed"] else "✗"
    print(f"  [{status}] {result['description']}")

# Test with another task to show generalization
register("sql query explanation",
    """You are a database engineer who specializes in explaining SQL to non-technical stakeholders.

Your task: Explain what the SQL query provided does in plain English.

Output format:
- One-sentence summary (what the query returns)
- Step-by-step breakdown (bullet points, one per clause)
- Business meaning (what this data is used for)

Constraints: Maximum 200 words. No SQL jargon — assume the reader knows only basic Excel.

Example:
Query: SELECT name, SUM(revenue) FROM sales GROUP BY name ORDER BY SUM(revenue) DESC LIMIT 10
Explanation: This query finds the top 10 salespeople by total revenue.

Now explain this SQL query:
{sql_query}""")

print("\n" + "=" * 50)
sql_prompt = generate_prompt("Explain a SQL query to a non-technical business stakeholder")
sql_quality = evaluate_prompt_quality(sql_prompt)
print(f"SQL task prompt quality: {sql_quality['score']}/{sql_quality['max']}")
for name, result in sql_quality["checks"].items():
    status = "✓" if result["passed"] else "✗"
    print(f"  [{status}] {result['description']}")


### What just happened?
- **Rubric scoring** gives objective quality measurements — not subjective "does it feel right" review.
- The meta-prompt **generalizes** to a different domain (SQL) and still scores well.
- The `{sql_query}` placeholder shows that the generated prompt is immediately usable as a template.


In [ ]:
# Challenge: Multi-step ReAct with two tool calls
# ─────────────────────────────────────────────────────────────
# Extend the ReAct agent to answer a question that requires TWO separate searches.
#
# Question: "What is the GDP of France and who founded OpenAI?"
# This requires: search('France GDP') AND search('OpenAI founded')
#
# Your tasks:
#   1. Register mock model responses for a 2-step ReAct chain:
#      - Step 1: Thought + search[France GDP]
#      - Step 2: Thought + search[OpenAI founded]
#      - Step 3: Final Answer combining both observations
#   2. Run react_agent() with max_steps=5 on the multi-part question
#   3. Implement a ToT variant: at step 1, generate 2 candidate search queries,
#      score them (which gives more useful info first?), and pick the best
#   4. Print the full reasoning trace showing Thought, Action, Observation for each step
#
# Starter:
MULTI_QUESTION = "What is the GDP of France and who founded OpenAI?"

# TODO: register step 1, 2, 3 model responses
# register("gdp of france", "Thought: ...\nAction: search[France GDP]")
# register("observation: france's gdp", "Thought: ...\nAction: search[OpenAI founded]")
# register("observation: openai was founded", "Thought: I have both answers.\nFinal Answer: ...")

# answer = react_agent(MULTI_QUESTION, max_steps=5)


---
## Day 6 key concepts recap
| Concept | What to remember |
|---|---|
| ReAct | Thought forces planning before action — without it, tool calls are random |
| Parse step | Use regex to extract Thought/Action/FinalAnswer from free text; prefer function calling in production |
| Tree of Thought | Generate N candidates, score each, select best — 3–5× more expensive but finds better solutions |
| ToT scoring | A separate LLM call per candidate with explicit criteria; numerical scores enable easy comparison |
| Meta-prompting | Input: task description. Output: ready-to-use prompt with role, format, constraint, example |
| Rubric evaluation | Score generated prompts programmatically — don't trust LLM output without measurement |

> **Tip:** ReAct is the foundation of most production agents. The Thought step forces the model to plan before acting; without it, tool calls are random rather than intentional.

---
## What's next
**Day 7** → Capstone — Prompt System Design and Evaluation: build a full multi-variant prompt suite with LLM-as-judge scoring and a decision guide.

Mark Day 6 complete in your [tracker](../index.html).
